# 08. Paoay: prepare Sentinel-2 inputs

**One task:** prepare eight half-month composites for a chosen four-month period over notebook 07’s study boundary. Existing verified files are reused; only missing files can be downloaded.

Default: **January–April 2024**, selected after the clear-coverage audit. The notes describe January–February cabbage followed by carrot in B1, and a carrot-to-potato transition in Area 2. This is a seasonal demonstration, not confirmed 2024 ground truth or a single crop cycle. Growth-stage alignment with external training remains unverified.

No Dynamic World, feature calculation, model fitting, crop prediction or accuracy evaluation happens here. See the [Paoay guide](../../docs/paoay_workflow.md).


## 1. Setup and period

Run notebook 07 first. Default `imagery_mode = "download_missing"` retrieves missing matching files and reuses verified files. Choose `"reuse"` for local-only inspection. Direct downloads create no Earth Engine export jobs; packages and local settings are unchanged.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import imagery
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

period_start = "2024-01-01"  # January–April 2024; end exclusive: 2024-05-01.
imagery_mode = "download_missing"      # Download only missing files; use "reuse" for local-only inspection.
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
reference_pointer = paoay_root / "references/reference_stage.json"
imagery_root = paoay_root / "sentinel2"
imagery_pointer = imagery_root / "imagery_stage.json"
write_stage_pointer(imagery_pointer, status="pending", reason="Notebook 08 has not finished the current imagery request")

stages = ["Plan the period and grid", "Check local imagery", "Prepare matching composites", "Verify imagery handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "08_paoay_sentinel2", stages, report_folder.parent / "progress",
    {"start": period_start, "mode": imagery_mode},
)


## 2. Plan the period and grid

Use the verified study boundary, snapped outward to a 10 m grid in its local projected CRS. No extra parcel buffer is added. The rectangular TIFFs include a margin outside the boundary; coverage statistics use only pixel centres inside it.


In [ ]:
pipeline_progress.begin_stage("Plan the period and grid")
reference_handoff = resolve_stage_pointer(reference_pointer)
request = imagery.prepare_request(reference_handoff, period_start, imagery_root)
print("Period:", request["period"]["start"], "to", request["period"]["end"], "(end exclusive)")
print("Grid CRS:", request["settings"]["grid"]["crs"])
print("Pixel size: 10 m; composites: 8")
print("Request folder:", request["folder"])
show_table(request["period"]["intervals"], title="Half-month inputs")
pipeline_progress.end_stage()


## 3. Check local imagery

Only files with matching geometry, dates and processing receipts are reused. Legacy Cloud Score+ TIFFs and external training archives are left untouched. Changed or incomplete cached pairs require inspection and are never silently overwritten.


In [ ]:
pipeline_progress.begin_stage("Check local imagery")
local_status = imagery.cache_status(request)
show_table(local_status, title="Matching local imagery")
missing_composites = sum(row["status"] == "missing" for row in local_status)
print("Missing composites:", missing_composites)
pipeline_progress.end_stage()


## 4. Prepare matching composites

The shared training recipe uses Sentinel-2 SR Harmonized, exact cloud-probability matches, probability < 40, SCL/edge masks and all six valid bands. It mosaics observations from the same date before each half-month median. B5/B11 use bilinear resampling; masks retain nearest-neighbour handling.

Each TIFF stores **B3, B4, B5, B8, B11, B2, clear_count**. Reflectance stays unscaled (apply 0.0001 later); clear counts are never scaled. Empty bins retain nodata and zero counts. No gap filling or quality thresholds are relaxed.


In [ ]:
pipeline_progress.begin_stage("Prepare matching composites")
earth_engine = None
if imagery_mode == "download_missing" and missing_composites:
    project = local_settings.get("earth_engine_project", "").strip()
    if not project or project == "YOUR_EARTH_ENGINE_PROJECT":
        raise ValueError("Set your Earth Engine project in settings.local.json before downloading")
    from crop_pipeline.notebook_setup import configure_certificates
    configure_certificates()
    import ee
    try:
        ee.Initialize(project=project)
    except ee.EEException:
        ee.Authenticate()
        ee.Initialize(project=project)
    earth_engine = ee

def composite_progress(items):
    return pipeline_progress.track(items, "Prepare Sentinel-2 composites", unit="composite")

imagery_handoff = imagery.collect(request, mode=imagery_mode, earth_engine=earth_engine, iterator=composite_progress)
pipeline_progress.end_stage()


## 5. Verify and save the handoff

A successful handoff requires all eight matching, checksum-verified raster/receipt pairs. An empty composite is a completed retrieval with insufficient observations, not a usable crop input. Later feature/model gates decide usable pixels.


In [ ]:
pipeline_progress.begin_stage("Verify imagery handoff")
image_bundle = imagery.load_imagery_bundle(imagery_handoff)
quality_rows = []
for interval in request["period"]["intervals"]:
    record = imagery.cached_interval(request, interval)
    quality_rows.append({"interval": interval["name"], **record["quality"]})
show_table(quality_rows, title="Clear observations inside the study boundary")
summary = image_bundle["manifest"]["summary"]
print("ROI pixels:", summary["roi_pixels"])
print("Pixels clear in all eight bins:", summary["all_eight_bins_clear_pixels"])
print("All-eight-bin clear coverage:", f"{summary['all_eight_bins_clear_fraction']:.1%}")
print("Empty half-month bins:", summary["empty_bins"])
write_stage_pointer(imagery_pointer, imagery_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

The image dates describe when the satellite observed the site. They do not establish planting dates or verify the KMZ crop notes.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified imagery handoff:", imagery_handoff)
print("Current imagery pointer:", imagery_pointer)
print("Calendar demonstration:", request["period"]["start"], "to", request["period"]["end"], "(end exclusive)")
print("Coverage is not crop accuracy. Crop growth-stage alignment remains unverified.")
print("Next: notebook 09 will obtain matching Dynamic World inputs in a separate batch.")
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
